# FRED Time Series — MCP & DataFetcherAgent Tests

Tests the MCP interface directly and via the `DataFetcherAgent`.

In [1]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env
from langchain_mcp_adapters.client import MultiServerMCPClient

set_anthropic_env(filedir='../../.keys')

## 1. Direct MCP Tool Call

Verify the MCP server is reachable and `fred.series_observations` returns expected data.

In [2]:
from lib.mcp_client import MCPClient, MCPClientConfig
from apps.agentic.core.constants import MCP_URL

mcp_config = MCPClientConfig(url=MCP_URL)

async def call_tool(tool_name, arguments=None):
    async with MCPClient(mcp_config) as client:
        return await client.call_tool(tool_name, arguments or {})

async def list_mcp_tools():
    async with MCPClient(mcp_config) as client:
        return await client.list_tools()

tools = await list_mcp_tools()
for t in tools:
    print(f'{t.name}: {t.description}')

fred_category_children: List the child categories for a FRED category.
fred_category_series: List the series contained within a FRED category.
fred_series_info: Fetch metadata for a single FRED series.
fred_series_observations: Return observations for a series (limit defaults to 100).
fred_series_updates: Return recently updated FRED series.
fred_release_series: List the series that belong to a FRED release.


In [3]:
SERIES_ID = 'BOGMBBM'

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 5})

print(result)
if result.structuredContent:
    data = result.structuredContent['result']
    print(f"count  : {data['count']}")
    print(f"limit  : {data['limit']}")
    print(f"offset : {data['offset']}")
    print()
    for obs in data['observations']:
        print(obs)

meta=None content=[TextContent(type='text', text='{\n  "realtime_start": "2026-04-04",\n  "realtime_end": "2026-04-04",\n  "order_by": "observation_date",\n  "sort_order": "asc",\n  "count": 806,\n  "offset": 0,\n  "limit": 5,\n  "observations": [\n    {\n      "realtime_start": "2026-04-04",\n      "realtime_end": "2026-04-04",\n      "date": "1959-01-01",\n      "value": "18.9"\n    },\n    {\n      "realtime_start": "2026-04-04",\n      "realtime_end": "2026-04-04",\n      "date": "1959-02-01",\n      "value": "18.6"\n    },\n    {\n      "realtime_start": "2026-04-04",\n      "realtime_end": "2026-04-04",\n      "date": "1959-03-01",\n      "value": "18.4"\n    },\n    {\n      "realtime_start": "2026-04-04",\n      "realtime_end": "2026-04-04",\n      "date": "1959-04-01",\n      "value": "18.7"\n    },\n    {\n      "realtime_start": "2026-04-04",\n      "realtime_end": "2026-04-04",\n      "date": "1959-05-01",\n      "value": "18.6"\n    }\n  ]\n}', annotations=None, meta=None)

In [4]:
from datetime import datetime

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 10000})

if result.structuredContent:
    observations = result.structuredContent['result']['observations']

    timestamps = []
    values = []
    for obs in observations:
        if obs['value'] == '.':
            continue
        timestamps.append(datetime.strptime(obs['date'], '%Y-%m-%d'))
        values.append(float(obs['value']))

    print(f'Series  : {SERIES_ID}')
    print(f'Points  : {len(timestamps)}')
    print(f'Start   : {timestamps[0].date()}')
    print(f'End     : {timestamps[-1].date()}')
    print(f'Min val : {min(values):.4f}')
    print(f'Max val : {max(values):.4f}')

Series  : BOGMBBM
Points  : 806
Start   : 1959-01-01
End     : 2026-02-01
Min val : 12.3000
Max val : 4193.2000


## 2. MCP Tool Discovery via `MultiServerMCPClient`

Verify that `langchain-mcp-adapters` discovers tools from the MCP server correctly.

In [5]:
client = MultiServerMCPClient({'fred': {'transport': 'sse', 'url': MCP_URL}})
discovered_tools = await client.get_tools()

print(f'Discovered {len(discovered_tools)} tools:')
for t in discovered_tools:
    print(f'  {t.name}: {t.description}') 

Discovered 6 tools:
  fred_category_children: List the child categories for a FRED category.
  fred_category_series: List the series contained within a FRED category.
  fred_series_info: Fetch metadata for a single FRED series.
  fred_series_observations: Return observations for a series (limit defaults to 100).
  fred_series_updates: Return recently updated FRED series.
  fred_release_series: List the series that belong to a FRED release.


## 3. DataFetcherAgent — Direct Invocation

Create the agent via `DataFetcherAgent.create()` and invoke it directly.

In [15]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
import shortuuid

from apps.agentic.agents.data.data_fetcher_agent import DataFetcherAgent

agent = await DataFetcherAgent.create()
print(f'Agent initialized with tools: {list(agent.tool_node.tools_by_name.keys())}')

DEBUG:    DataFetcherAgent discovered 6 MCP tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'fred_release_series']
DEBUG:    DataFetcherAgent prompt: 
            <instructions>
            You are a data fetcher agent. Use the available MCP tools to retrieve time series data
            from external data sources. Return the raw tool result unmodified — do not summarize,
            reformat, or interpret it. The result will be consumed by downstream agents for
            plotting or analysis.

            When fetching FRED data use the fred_series_observations tool with a limit of 
            10000 to retrieve the full series.
            </instructions>
            
Agent initialized with tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'fred_release_series']


In [16]:
state = {'messages': [HumanMessage(content=f'Fetch the {SERIES_ID} series from FRED with a limit of 10000.')]}
run_config = RunnableConfig(configurable={'thread_id': shortuuid.uuid()})

result = await agent.agent.ainvoke(state, run_config)
response = result['messages'][-1].content
print(response[:500])

INFO:     [DataFetcherAgent] routing → fred_series_observations({'series_id': 'BOGMBBM', 'limit': 10000})
Here is the raw result for the **BOGMBBM** series (Board of Governors, Monetary Base; Billions of Dollars, Monthly, Not Seasonally Adjusted) retrieved from FRED.

**Key details from the response:**

- **Series ID:** BOGMBBM
- **Total observations returned:** 806
- **Date range:** 1959-01-01 through 2026-02-01
- **Limit applied:** 10,000 (all 806 observations retrieved in a single call)
- **Sort order:** Ascending by observation date
- **Realtime as of:** 2026-04-04

**Notable values:**

- **1959
